<a href="https://colab.research.google.com/github/chandraniraychowdhury5/DS/blob/main/Wind_EnergyRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **This project incorporates AI-assisted development to improve productivity.**
# **Feedback or Suggestions are welcome**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install catboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

from sklearn.svm import SVR

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Create Output Folders
# ============================================================

folders = [
    "Regression_Results",
    "Regression_Results/Models",
    "Regression_Results/Predictions",
    "Regression_Results/Figures",
    "Regression_Results/FeatureImportance"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully.")

# ============================================================
# Load Dataset
# ============================================================

df = pd.read_csv("/content/drive/MyDrive/datasetAndProgram/WindDatasetForPrediction.csv")
print(df.shape)
print(df.head())

# ============================================================
# Remove Missing Values
# ============================================================

print("\nMissing Values")
print(df.isnull().sum())

df.dropna(inplace=True)

print("\nDataset Shape after removing missing values")
print(df.shape)

# ============================================================
# Remove Duplicate Rows
# ============================================================

df.drop_duplicates(inplace=True)

print("\nDataset Shape after removing duplicates")
print(df.shape)

# ============================================================
# Remove Date/Time Column if Present
# ============================================================

remove_columns = []

for col in df.columns:

    if "time" in col.lower():

        remove_columns.append(col)

if len(remove_columns) > 0:

    print("\nRemoving Columns")
    print(remove_columns)

    df.drop(columns=remove_columns, inplace=True)

# ============================================================
# Define Target Variable
# ============================================================

TARGET = "Power"

# ============================================================
# Feature Matrix and Target Vector
# ============================================================

X = df.drop(columns=[TARGET])

y = df[TARGET]

print("\nNumber of Features :", X.shape[1])

print("Target :", TARGET)

# ============================================================
# Train Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("\nTraining Samples :", X_train.shape)

print("Testing Samples :", X_test.shape)

# ============================================================
# Standard Scaling
# Required for:
# Linear Regression
# Ridge
# Lasso
# ElasticNet
# SVR
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# Save Scaler
joblib.dump(scaler, "Regression_Results/Models/StandardScaler.pkl")
print("\nScaler Saved Successfully")

# ============================================================
# Model Comparison DataFrame
# ============================================================

results = pd.DataFrame(
    columns=[
        "Model",
        "MAE",
        "MSE",
        "RMSE",
        "R2",
        "Training_Time",
        "Prediction_Time"
    ]
)

print("\nRegression Pipeline Ready.")

Folders created successfully.
(43791, 17)
                  Time  temperature_2m  relativehumidity_2m  dewpoint_2m  \
0  2017-01-02 06:00:00            26.7                   92         24.9   
1  2017-01-02 07:00:00            28.4                   91         26.1   
2  2017-01-02 08:00:00            30.2                   88         27.0   
3  2017-01-02 09:00:00            34.0                   82         29.1   
4  2017-01-02 10:00:00            35.6                   73         27.7   

   windspeed_10m  winddirection_10m  windgusts_10m  Wind_Shear_Coefficient  \
0           2.10                 65            4.7                0.442704   
1           3.10                 69            4.8                0.303823   
2           3.54                 74            5.8                0.289216   
3           3.44                 82            6.5                0.242316   
4           3.71                 93            8.0                0.156538   

   Air_Density  Wind_Veering    

The saved StandardScaler.pkl contains these learned parameters:

Mean (mean_) of each feature

Standard deviation (scale_) of each feature

Variance (var_) of each feature

Number of features (n_features_in_)

Feature names (if available)

Other internal configuration parameters


MAE (Mean Absolute Error): The average of the absolute differences between actual and predicted values; lower is better.

MSE (Mean Squared Error): The average of the squared differences between actual and predicted values, giving more weight to larger errors; lower is better.

RMSE (Root Mean Squared Error): The square root of MSE, representing the typical prediction error in the original units of the target variable; lower is better.

R^2 (Coefficient of Determination): The proportion of the variance in the target variable explained by the model, measuring its goodness of fit; higher is better (closer to 1).

In [ ]:
# ============================================================
# Generic Regression Training Function
# ============================================================

def train_and_evaluate(model, model_name, scaled=False, feature_importance=False):

    print("\n" + "=" * 70)
    print(f"Training : {model_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # Select Dataset
    # --------------------------------------------------------
    if scaled:
        Xtr = X_train_scaled
        Xts = X_test_scaled
    else:
        Xtr = X_train
        Xts = X_test

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    start_train = time.time()
    model.fit(Xtr, y_train)
    end_train = time.time()
    training_time = end_train - start_train

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------
    start_predict = time.time()
    predictions = model.predict(Xts)
    end_predict = time.time()
    prediction_time = end_predict - start_predict

    # --------------------------------------------------------
    # Evaluation Metrics
    # --------------------------------------------------------
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    # --------------------------------------------------------
    # Print Results
    # --------------------------------------------------------
    print(f"MAE   : {mae:.6f}")
    print(f"MSE   : {mse:.6f}")
    print(f"RMSE  : {rmse:.6f}")
    print(f"R²    : {r2:.6f}")
    print(f"Training Time : {training_time:.4f} sec")
    print(f"Prediction Time : {prediction_time:.4f} sec")

    # --------------------------------------------------------
    # Save Model
    # --------------------------------------------------------
    joblib.dump(model, f"Regression_Results/Models/{model_name}.pkl")
    print("Model Saved.")

    # --------------------------------------------------------
    # Save Predictions
    # --------------------------------------------------------
    prediction_df = pd.DataFrame({"Actual": y_test.values, "Predicted": predictions})
    prediction_df.to_csv(f"Regression_Results/Predictions/{model_name}_Predictions.csv", index=False)
    print("Predictions Saved.")

    # --------------------------------------------------------
    # Actual vs Predicted Plot
    # --------------------------------------------------------
    plt.figure(figsize=(8,6))
    plt.scatter(y_test, predictions, alpha=0.6)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
    plt.xlabel("Actual Power")
    plt.ylabel("Predicted Power")
    plt.title(f"{model_name} : Actual vs Predicted")
    plt.tight_layout()
    plt.savefig(f"Regression_Results/Figures/{model_name}_Actual_vs_Predicted.png", dpi=300)
    plt.close()

    # --------------------------------------------------------
    # Residual Plot
    # --------------------------------------------------------
    residuals = y_test - predictions
    plt.figure(figsize=(8,6))
    plt.scatter(predictions, residuals, alpha=0.6)
    plt.axhline(0, color='red', linestyle='--')
    plt.xlabel("Predicted")
    plt.ylabel("Residual")
    plt.title(f"{model_name} Residual Plot")
    plt.tight_layout()
    plt.savefig(f"Regression_Results/Figures/{model_name}_Residual.png", dpi=300)
    plt.close()

    # --------------------------------------------------------
    # Feature Importance
    # --------------------------------------------------------
    if feature_importance:
        importance = model.feature_importances_
        feature_df = pd.DataFrame({"Feature": X.columns, "Importance": importance})
        feature_df = feature_df.sort_values(by="Importance", ascending=False)
        feature_df.to_csv(f"Regression_Results/FeatureImportance/{model_name}_Importance.csv", index=False)
        plt.figure(figsize=(10,8))
        sns.barplot(data=feature_df.head(20), x="Importance", y="Feature")
        plt.title(f"{model_name} Feature Importance")
        plt.tight_layout()
        plt.savefig(f"Regression_Results/FeatureImportance/{model_name}.png", dpi=300)
        plt.close()

    # --------------------------------------------------------
    # Save Results
    # --------------------------------------------------------
    global results
    results.loc[len(results)] = [model_name, mae, mse, rmse, r2, training_time, prediction_time]
    return model

In [ ]:
# ============================================================
# Define Models
# ============================================================

models = {
    # --------------------------------------------------------
    # Linear Models
    # --------------------------------------------------------
    "LinearRegression": (LinearRegression(), True, False),
    "Ridge": (Ridge(alpha=1.0), True, False),
    "Lasso": (Lasso(alpha=0.001), True, False),
    "ElasticNet": (ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42), True, False),

    # --------------------------------------------------------
    # Tree Models
    # --------------------------------------------------------
    "DecisionTree": (DecisionTreeRegressor(random_state=42), False, True),

    "RandomForest": (RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1), False, True),
    "ExtraTrees": (ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1), False, True),
    "GradientBoosting": (GradientBoostingRegressor(n_estimators=300, random_state=42), False, True),
    "AdaBoost": (AdaBoostRegressor(n_estimators=300, random_state=42), False, True),

    # --------------------------------------------------------
    # Kernel Model
    # --------------------------------------------------------
    "SVR": (SVR(kernel="rbf", C=10, gamma="scale"), True, False),

    # --------------------------------------------------------
    # Gradient Boosting Libraries
    # --------------------------------------------------------
    "XGBoost": (XGBRegressor(objective="reg:squarederror", n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1), False, True),
    "LightGBM": (LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42), False, True),
    "CatBoost": (CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6, random_seed=42, verbose=0), False, True)
}

print("=" * 70)
print("Total Regression Models :", len(models))
print("=" * 70)

# ============================================================
# Train All Models
# ============================================================
trained_models = {}
for model_name, (model, scaled, feature_importance) in models.items():
    trained_model = train_and_evaluate(model=model, model_name=model_name, scaled=scaled, feature_importance=feature_importance)
    trained_models[model_name] = trained_model

print("\n")
print("=" * 70)
print("All Models Trained Successfully")
print("=" * 70)

Total Regression Models : 13

Training : LinearRegression
MAE   : 0.024811
MSE   : 0.001143
RMSE  : 0.033813
R²    : 0.986002
Training Time : 0.0589 sec
Prediction Time : 0.0031 sec
Model Saved.
Predictions Saved.

Training : Ridge
MAE   : 0.024811
MSE   : 0.001143
RMSE  : 0.033813
R²    : 0.986002
Training Time : 0.0098 sec
Prediction Time : 0.0005 sec
Model Saved.
Predictions Saved.

Training : Lasso
MAE   : 0.024862
MSE   : 0.001172
RMSE  : 0.034233
R²    : 0.985652
Training Time : 0.0138 sec
Prediction Time : 0.0006 sec
Model Saved.
Predictions Saved.

Training : ElasticNet
MAE   : 0.024816
MSE   : 0.001157
RMSE  : 0.034014
R²    : 0.985836
Training Time : 0.0227 sec
Prediction Time : 0.0006 sec
Model Saved.
Predictions Saved.

Training : DecisionTree
MAE   : 0.033201
MSE   : 0.002214
RMSE  : 0.047052
R²    : 0.972896
Training Time : 1.2699 sec
Prediction Time : 0.0067 sec
Model Saved.
Predictions Saved.

Training : RandomForest
MAE   : 0.023202
MSE   : 0.001084
RMSE  : 0.032927
R²